In [1]:
from Bio import Entrez
import pandas as pd
import numpy as np
import os
import json
import time
import regex as re
import xmltodict, json
# ElementTree
import xml.etree.ElementTree as ET
from dataclasses import dataclass

from utils.utils import article_machine, fetch_pubmed_data_given_ids, Paper


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\20195435\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\20195435\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\20195435\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
def fetch_pubmed_data_given_ids_in_chunks(id_list, chunk_size=10000, output_path=None):
    papers_list = []
    
    for i in range(0, len(id_list), chunk_size):
        print(f"Fetching papers {i} to {min(i+chunk_size, len(id_list))} out of {len(id_list)}")
        chunk = id_list[i:i+chunk_size]

        try:
            papers = fetch_pubmed_data_given_ids(chunk)
        except Exception as e:
            print(f"Error fetching chunk {i}: {e}")
            continue

        # Process each paper
        for paper in papers.get('PubmedArticle', []):
            try:
                medline_citation = paper.get('MedlineCitation', {})
                article = medline_citation.get('Article', {})
                journal_info = article.get('Journal', {})
                journal_issue = journal_info.get('JournalIssue', {})
                pub_date = journal_issue.get('PubDate', {})

                # Extract data
                id = medline_citation.get('PMID', 'Not Available')
                
                # Skip if already exists
                if output_path:
                    filename = f"{output_path}/{id}.json"
                    if os.path.exists(filename):
                        print(f"Skipping {id} - already exists")
                        continue
                
                title = article.get('ArticleTitle', 'Not Available')
                abstract = article.get('Abstract', {}).get('AbstractText', 'Not Available')
                
                # Handle authors
                author_list = article.get('AuthorList', [])
                if isinstance(author_list, dict):
                    authors = author_list.get('Author', [])
                else:
                    authors = author_list
                authors = [f"{author.get('LastName', '')} {author.get('Initials', '')}".strip() 
                          for author in authors if isinstance(author, dict)]

                journal = journal_info.get('Title', 'Not Available')
                publication_year = pub_date.get('Year', 'Not Available')
                publication_month = pub_date.get('Month', 'Not Available')
                publication_day = pub_date.get('Day', 'Not Available')
                doi = article.get('ELocationID', 'Not Available')
                
                # Handle keywords - fixed to properly extract from various formats
                keywords = []
                keyword_list = medline_citation.get('KeywordList', [])
                if keyword_list:
                    for keyword_entry in keyword_list:
                        if isinstance(keyword_entry, dict):
                            kw_items = keyword_entry.get('Keyword', [])
                            if isinstance(kw_items, list):
                                for kw in kw_items:
                                    # Keywords can be strings or StringElement objects
                                    kw_text = str(kw) if kw else None
                                    if kw_text and kw_text != 'Not Available':
                                        keywords.append(kw_text)
                            elif kw_items:
                                keywords.append(str(kw_items))
                
                # If no keywords found, set default
                if not keywords:
                    keywords = ['Not Available']
                    
                language_list = article.get('Language', ['Not Available'])

                # Create Paper object
                paper_object = Paper(id, title, abstract, authors, journal, 
                                   publication_year, publication_month, publication_day, 
                                   doi, keywords, language_list)
                papers_list.append(paper_object)

                # Save to file
                if output_path and str(id).isdigit():
                    os.makedirs(output_path, exist_ok=True)
                    with open(filename, 'w', encoding='utf-8') as f:
                        json.dump(paper_object.__dict__, f, indent=2, default=str)
                    print(f"Saved paper {id}")
                    
            except Exception as e:
                print(f"Error processing paper: {e}")
                continue
            
        # Rate limiting
        time.sleep(0.5)
        
    return papers_list


In [24]:
# List of journals to include
journals_to_include = [
    "ACS Applied Materials & Interfaces",
    "ACS Nano",
    "Advanced Functional Materials",
    "Advanced Materials",
    "Angewandte Chemie",
    "Biology and Medicine",
    "Biomaterials",
    "Cell",
    "Clinical Cancer Research",
    "Frontiers in Nanotechnology",
    "Immunity",
    "International Journal of Nanomedicine",
    "Journal of Controlled Release",
    "Journal of Materials Chemistry B",
    "Matter",
    "Molecular Therapy",
    "Nano Letters",
    "Nano Micro Small",
    "Nano Research",
    "Nanomedicine",
    "Nanomedicine: Nanotechnology",
    "Nanoscale",
    "Nature",
    "Nature Biomedical Engineering",
    "Nature Cancer",
    "Nature Communications",
    "Nature Materials",
    "Nature Medicine",
    "Nature Nanotechnology",
    "NPG Asia Materials",
    "Pharmaceutics",
    "PNAS",
    "Science",
    "Science Advances",
    "Science Translational Medicine",
    "Scientific Reports",
    "Small"
]

# journals as small letters
journals_to_include = [journal.lower() for journal in journals_to_include]

# Inclusion criteria
keywords_inclusion = ["nanoparticles", "nanoparticle", "in vivo", "mouse model", "animal model"]

# Exclusion criteria
keywords_exclusion = ["review"]

In [25]:
# compile the query
base_query = f"({keywords_inclusion[0]} OR {keywords_inclusion[1]}) AND ({keywords_inclusion[2]} OR {keywords_inclusion[3]} OR {keywords_inclusion[4]})"

if False:
    # add the exclusion criteria
    query += f" AND NOT ({keywords_exclusion[0]})"

    # add the journals to include
    query += f" AND ("
    for journal in journals_to_include:
        query += f"source:{journal} OR "
    query = query[:-4]
    query += ")"

if False:
    # add the years to include 2015-2021
    query += f" AND (2015/01/01[Date - Publication]:2021/12/31[Date - Publication])"



In [26]:
base_query

'(nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model)'

In [27]:
# create a list of years from 1995 to 2026
years = list(range(2024, 2026))

In [28]:
years

[2024, 2025]

In [29]:
for year in years:
    query = base_query + f" AND ({year}/01/01[Date - Publication]:{year}/12/31[Date - Publication])"
    print(f"Querying for year {year}")
    print(f"Query: {query}")

    # search for the query
    id_list = article_machine(query)

    # print the number of results
    print(f"Number of results: {len(id_list)}")

    # fetch the data
    papers = fetch_pubmed_data_given_ids_in_chunks(id_list, output_path="papers")
    break
    

Querying for year 2024
Query: (nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model) AND (2024/01/01[Date - Publication]:2024/12/31[Date - Publication])
Total matching articles: 7013

Fetching 1 to 1000 out of 7013

Fetching 1001 to 2000 out of 7013

Fetching 2001 to 3000 out of 7013

Fetching 3001 to 4000 out of 7013

Fetching 4001 to 5000 out of 7013

Fetching 5001 to 6000 out of 7013

Fetching 6001 to 7000 out of 7013

Fetching 7001 to 7013 out of 7013

Retrieved 7013 article IDs
Number of results: 7013
Fetching papers 0 to 7013 out of 7013
{'MedlineCitation': DictElement({'GeneralNote': [], 'SpaceFlightMission': [], 'KeywordList': [], 'OtherAbstract': [], 'OtherID': [], 'InvestigatorList': [], 'CitationSubset': ['IM'], 'PMID': StringElement('39361752', attributes={'Version': '1'}), 'DateCompleted': {'Year': '2024', 'Month': '10', 'Day': '03'}, 'DateRevised': {'Year': '2025', 'Month': '04', 'Day': '17'}, 'Article': DictElement({'ArticleDate': [DictElement({'Yea

In [ ]:
len(os.listdir("papers"))

In [ ]:
len(os.listdir(r"/home/chris/Data/Projects/playground/nanotech_embed/embeddings"))